# 09 – EC3D Unknown Ablation Comparison

**Obiettivo**: Confrontare i risultati degli esperimenti EC3D con e senza la classe Unknown.

## Esperimenti confrontati:
1. **Fine-tuning**: Linear Probe, MLP Probe, Fine-tuning, From Scratch
2. **Zero-Shot Retrieval**: Generic Templates, FLAG3D Mean, FLAG3D Max
3. **One-Shot Retrieval**: Pose→Pose
4. **Few-Shot Prototypes**: k=1,2,5,10

In [ ]:
import json
from pathlib import Path
import pandas as pd
import numpy as np
from datetime import datetime

ROOT_DIR = Path('..').resolve()
RESULTS_DIR = ROOT_DIR / 'results' / 'ec3d'

WITH_UNKNOWN_DIR = RESULTS_DIR / 'WITH_UNKNOWN'
NO_UNKNOWN_DIR = RESULTS_DIR / 'NO_UNKNOWN'

print(f'WITH_UNKNOWN: {WITH_UNKNOWN_DIR}')
print(f'NO_UNKNOWN: {NO_UNKNOWN_DIR}')

In [ ]:
def load_json_results(path):
    if path.exists():
        with open(path) as f:
            return json.load(f)
    return None

## 1. Fine-tuning Comparison

In [ ]:
# Load fine-tuning results
ft_with = load_json_results(WITH_UNKNOWN_DIR / 'finetuning_metrics.json')
ft_no = load_json_results(NO_UNKNOWN_DIR / 'finetuning_metrics.json')

if ft_with and ft_no:
    methods = ['linear_probe', 'mlp_probe', 'finetune', 'from_scratch']
    ft_comparison = []
    
    for m in methods:
        with_res = ft_with['results'].get(m, {})
        no_res = ft_no['results'].get(m, {})
        
        for metric in ['top1_acc', 'macro_f1']:
            with_val = with_res.get(metric, None)
            no_val = no_res.get(metric, None)
            
            if with_val is not None and no_val is not None:
                delta = no_val - with_val
                delta_pct = (delta / with_val * 100) if with_val != 0 else 0
                ft_comparison.append({
                    'Method': m,
                    'Metric': metric,
                    'WITH_UNKNOWN': f'{with_val:.4f}',
                    'NO_UNKNOWN': f'{no_val:.4f}',
                    'Delta': f'{delta:+.4f}',
                    'Delta %': f'{delta_pct:+.2f}%'
                })
    
    ft_df = pd.DataFrame(ft_comparison)
    print('=' * 80)
    print('FINE-TUNING COMPARISON')
    print('=' * 80)
    print(ft_df.to_string(index=False))
else:
    print('Fine-tuning results not found. Run notebooks 05 first.')
    ft_df = None

## 2. Zero-Shot Retrieval Comparison

In [ ]:
# Load zero-shot results
zs_with = load_json_results(WITH_UNKNOWN_DIR / 'zero_shot_metrics.json')
zs_no = load_json_results(NO_UNKNOWN_DIR / 'zero_shot_metrics.json')

if zs_with and zs_no:
    strategies = ['generic_templates', 'flag3d_mean', 'flag3d_max']
    zs_comparison = []
    
    for s in strategies:
        with_res = zs_with['results'].get(s, {})
        no_res = zs_no['results'].get(s, {})
        
        for metric in ['recall@1', 'recall@5']:
            with_val = with_res.get(metric, None)
            no_val = no_res.get(metric, None)
            
            if with_val is not None and no_val is not None:
                delta = no_val - with_val
                delta_pct = (delta / with_val * 100) if with_val != 0 else float('inf') if delta > 0 else 0
                zs_comparison.append({
                    'Strategy': s,
                    'Metric': metric,
                    'WITH_UNKNOWN': f'{with_val:.4f}',
                    'NO_UNKNOWN': f'{no_val:.4f}',
                    'Delta': f'{delta:+.4f}',
                    'Delta %': f'{delta_pct:+.2f}%' if delta_pct != float('inf') else '+∞'
                })
    
    zs_df = pd.DataFrame(zs_comparison)
    print('\n' + '=' * 80)
    print('ZERO-SHOT RETRIEVAL COMPARISON')
    print('=' * 80)
    print(zs_df.to_string(index=False))
else:
    print('Zero-shot results not found. Run notebooks 06 first.')
    zs_df = None

## 3. One-Shot Retrieval Comparison

In [ ]:
# Load one-shot results
os_with = load_json_results(WITH_UNKNOWN_DIR / 'one_shot_metrics.json')
os_no = load_json_results(NO_UNKNOWN_DIR / 'one_shot_metrics.json')

if os_with and os_no:
    with_acc = os_with['results']['mean_accuracy']
    with_std = os_with['results']['std_accuracy']
    no_acc = os_no['results']['mean_accuracy']
    no_std = os_no['results']['std_accuracy']
    
    delta = no_acc - with_acc
    delta_pct = (delta / with_acc * 100) if with_acc != 0 else 0
    
    print('\n' + '=' * 80)
    print('ONE-SHOT RETRIEVAL COMPARISON')
    print('=' * 80)
    print(f'WITH_UNKNOWN: {with_acc:.4f} ± {with_std:.4f}')
    print(f'NO_UNKNOWN:   {no_acc:.4f} ± {no_std:.4f}')
    print(f'Delta:        {delta:+.4f} ({delta_pct:+.2f}%)')
else:
    print('One-shot results not found. Run notebooks 07 first.')

## 4. Few-Shot Prototypes Comparison

In [ ]:
# Load few-shot results
fs_with = load_json_results(WITH_UNKNOWN_DIR / 'few_shot_metrics.json')
fs_no = load_json_results(NO_UNKNOWN_DIR / 'few_shot_metrics.json')

if fs_with and fs_no:
    fs_comparison = []
    
    for with_row in fs_with['summary']:
        k = with_row['k']
        no_row = next((r for r in fs_no['summary'] if r['k'] == k), None)
        
        if no_row:
            with_acc = with_row['acc_mean']
            no_acc = no_row['acc_mean']
            delta = no_acc - with_acc
            delta_pct = (delta / with_acc * 100) if with_acc != 0 else 0
            
            fs_comparison.append({
                'k': k,
                'WITH_UNKNOWN (acc ± std)': f"{with_acc:.4f} ± {with_row['acc_std']:.4f}",
                'NO_UNKNOWN (acc ± std)': f"{no_acc:.4f} ± {no_row['acc_std']:.4f}",
                'Delta': f'{delta:+.4f}',
                'Delta %': f'{delta_pct:+.2f}%'
            })
    
    fs_df = pd.DataFrame(fs_comparison)
    print('\n' + '=' * 80)
    print('FEW-SHOT PROTOTYPES COMPARISON')
    print('=' * 80)
    print(fs_df.to_string(index=False))
else:
    print('Few-shot results not found. Run notebooks 08 first.')
    fs_df = None

## 5. Export Summary

In [ ]:
# Create combined summary
summary_data = []

# Add fine-tuning
if ft_with and ft_no:
    for m in ['linear_probe', 'mlp_probe', 'finetune', 'from_scratch']:
        with_val = ft_with['results'].get(m, {}).get('top1_acc', 0)
        no_val = ft_no['results'].get(m, {}).get('top1_acc', 0)
        summary_data.append({
            'Experiment': 'Fine-tuning',
            'Method': m,
            'WITH_UNKNOWN': with_val,
            'NO_UNKNOWN': no_val,
            'Delta': no_val - with_val
        })

# Add zero-shot
if zs_with and zs_no:
    for s in ['generic_templates', 'flag3d_mean', 'flag3d_max']:
        with_val = zs_with['results'].get(s, {}).get('recall@1', 0)
        no_val = zs_no['results'].get(s, {}).get('recall@1', 0)
        summary_data.append({
            'Experiment': 'Zero-Shot',
            'Method': s,
            'WITH_UNKNOWN': with_val,
            'NO_UNKNOWN': no_val,
            'Delta': no_val - with_val
        })

# Add one-shot
if os_with and os_no:
    summary_data.append({
        'Experiment': 'One-Shot',
        'Method': 'pose2pose',
        'WITH_UNKNOWN': os_with['results']['mean_accuracy'],
        'NO_UNKNOWN': os_no['results']['mean_accuracy'],
        'Delta': os_no['results']['mean_accuracy'] - os_with['results']['mean_accuracy']
    })

# Add few-shot
if fs_with and fs_no:
    for with_row in fs_with['summary']:
        k = with_row['k']
        no_row = next((r for r in fs_no['summary'] if r['k'] == k), None)
        if no_row:
            summary_data.append({
                'Experiment': 'Few-Shot',
                'Method': f'k={k}',
                'WITH_UNKNOWN': with_row['acc_mean'],
                'NO_UNKNOWN': no_row['acc_mean'],
                'Delta': no_row['acc_mean'] - with_row['acc_mean']
            })

summary_df = pd.DataFrame(summary_data)

if len(summary_df) > 0:
    # Save CSV
    summary_df.to_csv(RESULTS_DIR / 'ec3d_unknown_ablation_summary.csv', index=False)
    
    # Save Markdown
    md_content = f'''# EC3D Unknown Ablation Summary

Generated: {datetime.now().isoformat()}

## Overview

Comparison of EC3D experiments with and without the Unknown class (SQUAT, instruction_id=10).

- **WITH_UNKNOWN**: 12 classes (includes Unknown)
- **NO_UNKNOWN**: 11 classes (paper-aligned, excludes Unknown)

## Results

{summary_df.to_markdown(index=False)}

## Key Observations

- Positive Delta indicates improvement when removing Unknown class
- Negative Delta indicates worse performance without Unknown class
'''
    
    with open(RESULTS_DIR / 'ec3d_unknown_ablation_summary.md', 'w') as f:
        f.write(md_content)
    
    print('\n' + '=' * 80)
    print('SUMMARY SAVED')
    print('=' * 80)
    print(f'CSV: {RESULTS_DIR / "ec3d_unknown_ablation_summary.csv"}')
    print(f'MD:  {RESULTS_DIR / "ec3d_unknown_ablation_summary.md"}')
else:
    print('No results found to summarize.')

In [ ]:
# Final summary
print('\n' + '=' * 80)
print('ABLATION STUDY COMPLETE')
print('=' * 80)
print('\nThis notebook compares all EC3D experiments with and without the Unknown class.')
print('\nTo run the full ablation study:')
print('  1. Run notebooks 05-08 for WITH_UNKNOWN version (original data)')
print('  2. Run notebooks 05-08 _no_unknown versions')
print('  3. Re-run this notebook to generate comparison')